In [5]:
import pandas as pd
import os
import math
import json

# Data preparation

## Yoruba

Create symlink

```bash
ln -s /network/scratch/g/guzmand/datasets/bible-tts-resources/Yoruba/wav /home/mila/g/guzmand/scratch/Repositories/F5-TTS/data/open-bible-yoruba/wavs
```

Create metadata

In [ ]:
DATASET_PATH = "/network/scratch/g/guzmand/datasets/bible-tts-resources/Yoruba"

train = pd.read_csv(os.path.join(DATASET_PATH, "train.tsv"), sep="\t")
train = train[["filename", "text"]]
train.columns = ["audio_file", "text"]
train["audio_file"] = train["audio_file"].apply(lambda x: os.path.join("wavs", x))
train.to_csv("data/open-bible-yoruba/metadata.csv", index=False, sep="|")

,audio_file,text
0,wavs/New-Testament-1-Corinthians-001-001.wav,"Paulu, ẹni ti a pé láti jẹ́ aposteli Kristi Je..."
1,wavs/New-Testament-1-Corinthians-001-003.wav,Oore-ọ̀fẹ́ àti àlàáfíà fún yín láti ọ̀dọ̀ Ọlọ́...
2,wavs/New-Testament-1-Corinthians-001-004.wav,Nígbà gbogbo ni mo ń dúpẹ́ lọ́wọ́ Ọlọ́run fún ...
3,wavs/New-Testament-1-Corinthians-001-005.wav,Nítorí nínú rẹ̀ ni a ti sọ yín di ọlọ́rọ̀ nínú...
4,wavs/New-Testament-1-Corinthians-001-006.wav,Nítorí ẹ̀rí wa nínú Kristi ni a ti fi ìdí rẹ̀ ...
...,...,...
24424,wavs/Old-Testament-Zephaniah-003-015.wav,"Olúwa ti mú ìdájọ́ rẹ wọ̀n-ọn-nì kúrò, ó sì ti..."
24425,wavs/Old-Testament-Zephaniah-003-016.wav,"Ní ọjọ́ náà, wọn yóò sọ fún Jerusalẹmu pé, “Má..."
24426,wavs/Old-Testament-Zephaniah-003-017.wav,"Olúwa Ọlọ́run rẹ wà pẹ̀lú rẹ, Ó ní agbára láti..."
24427,wavs/Old-Testament-Zephaniah-003-018.wav,“Èmi ó kó àwọn tí ó ń banújẹ́ fún àjọ̀dún tí a...


Preprocess the data
```bash
python src/f5_tts/train/datasets/prepare_csv_wavs.py \
    data/open-bible-yoruba \
    data/open-bible-yoruba_pinyin \
    --workers 4
```

Verify the vocabulary

In [2]:
with open("data/open-bible-yoruba_pinyin/vocab.txt", "r") as f:
    vocab = [line.rstrip("\n") for line in f]

print(f"Vocab size: {len(vocab)}")
print(f"First entry is space: {repr(vocab[0])}")  # Must be ' '

# Check Yoruba-specific characters are present
yoruba_chars = ['ẹ', 'ọ', 'ṣ', 'Ẹ', 'Ọ', 'Ṣ', 'ń', 'Ń', 'ǹ', 'Ǹ']
combining_marks = ['\u0300', '\u0301']  # combining grave, combining acute

for c in yoruba_chars + combining_marks:
    status = "✓" if c in vocab else "✗ MISSING"
    print(f"  {repr(c):10s} → {status}")

Vocab size: 96
First entry is space: ' '
  'ẹ'        → ✓
  'ọ'        → ✓
  'ṣ'        → ✓
  'Ẹ'        → ✓
  'Ọ'        → ✓
  'Ṣ'        → ✓
  'ń'        → ✓
  'Ń'        → ✓
  'ǹ'        → ✓
  'Ǹ'        → ✓
  '̀'        → ✓
  '́'        → ✓


Create the training config

```bash
cp src/f5_tts/configs/F5TTS_v1_Base.yaml src/f5_tts/configs/F5TTS_v1_Base_Open_Bible_Yoruba.yaml
```

A few parameters need to be changed:
- batch_size_per_gpu: 28000 (calculate it for your dataset, give it room)
- epochs: 670 (calculate it for your dataset)
- max_samples: 32
- num_workers: 4
- save_per_updates: 10000
- keep_last_n_checkpoints: 5



In [8]:
with open('data/open-bible-yoruba_pinyin/duration.json') as f:
    d = json.load(f)['duration']
print(f'Total samples: {len(d)}')
print(f'Total hours: {sum(d)/3600:.2f}')
print(f'Avg duration: {sum(d)/len(d):.2f}s')
print(f'Min duration: {min(d):.2f}s')
print(f'Max duration: {max(d):.2f}s')
avg_frames = (sum(d)/len(d)) * 24000 / 256
print(f'Avg frames per sample: {avg_frames:.0f}')
print(f'Frame budget for ~32 samples: {32 * avg_frames:.0f}')

Total samples: 24429
Total hours: 61.92
Avg duration: 9.12s
Min duration: 1.65s
Max duration: 15.00s
Avg frames per sample: 855
Frame budget for ~32 samples: 27375


In [11]:
# Dataset info
with open('data/open-bible-yoruba_pinyin/duration.json') as f:
    d = json.load(f)
durs = d['duration']

total_samples = len(durs)
total_duration = sum(durs)  # in seconds

# Config values (must match your .yaml)
batch_size_per_gpu = 28000  # frames_threshold
hop_length = 256
sampling_rate = 24000
grad_accumulation_steps = 1
num_gpus = 1  # adjust if using multi-gpu
max_samples = 32  # max sequences per batch

# Simulate DynamicBatchSampler: sort by frame length, greedy bin-pack
frames_per_second = sampling_rate / hop_length  # 93.75
frame_lens = sorted([dur * frames_per_second for dur in durs])

num_batches = 0
batch_frames = 0
batch_count = 0
for frame_len in frame_lens:
    if batch_frames + frame_len <= batch_size_per_gpu and (max_samples == 0 or batch_count < max_samples):
        batch_frames += frame_len
        batch_count += 1
    else:
        if batch_count > 0:
            num_batches += 1
        if frame_len <= batch_size_per_gpu:
            batch_frames = frame_len
            batch_count = 1
        else:
            batch_frames = 0
            batch_count = 0
if batch_count > 0:
    num_batches += 1

total_frames = total_duration * frames_per_second
updates_per_epoch = math.ceil(num_batches / (num_gpus * grad_accumulation_steps))

target_updates = 500_000
epochs = target_updates / updates_per_epoch

print(f'Total samples: {total_samples}')
print(f'Total duration: {total_duration:.0f}s ({total_duration/3600:.2f} hours)')
print(f'Total frames: {total_frames:.0f}')
print(f'Frames per second: {frames_per_second}')
print(f'')
print(f'batch_size_per_gpu: {batch_size_per_gpu} frames')
print(f'max_samples: {max_samples}')
print(f'GPUs: {num_gpus}')
print(f'grad_accumulation_steps: {grad_accumulation_steps}')
print(f'')
print(f'Batches from DynamicBatchSampler: {num_batches}')
print(f'Avg frames per batch: {total_frames/num_batches:.0f} / {batch_size_per_gpu} ({total_frames/num_batches/batch_size_per_gpu*100:.1f}% utilization)')
print(f'Estimated updates per epoch: {updates_per_epoch}')
print(f'Target updates: {target_updates}')
print(f'Required epochs: {epochs:.1f} → ceil to {math.ceil(epochs)}')

Total samples: 24429
Total duration: 222914s (61.92 hours)
Total frames: 20898166
Frames per second: 93.75

batch_size_per_gpu: 28000 frames
max_samples: 32
GPUs: 1
grad_accumulation_steps: 1

Batches from DynamicBatchSampler: 870
Avg frames per batch: 24021 / 28000 (85.8% utilization)
Estimated updates per epoch: 870
Target updates: 500000
Required epochs: 574.7 → ceil to 575


In [12]:
870*575

500250

In [14]:
(13 * 575) / 60

124.58333333333333

Train from scratch
```bash
accelerate launch --mixed_precision bf16 src/f5_tts/train/train.py --config-name F5TTS_v1_Base_Open_Bible_Yoruba.yaml
```

In [10]:
870 * 670

582900

## Iroyinspeech

In [ ]:
path = "/home/mila/g/guzmand/scratch/datasets/IroyinSpeech/male/train_5_hours.tsv"
df = pd.read_csv(path, sep="\t", names=["audio_file", "text"])

samples_train = len(df)
batch_size = 16
steps_per_epoch = math.ceil(samples_train / batch_size)
total_steps = 100_000

total_epochs = total_steps // steps_per_epoch
print(f"Total epochs: {total_epochs}")

df["audio_file"] = "wavs/" + df["audio_file"]
df.to_csv("data/yoruba-5-hours/metadata.csv", index=False, sep="|")
df

Total epochs: 375


,audio_file,text
0,wavs/yo_p_0001.wav,Mo pàdé alàgbà ìnàkí yìí nílé ìtur...
1,wavs/yo_p_0002.wav,"Ilé ìtura náà ti gbó, tàbí ká ní ó t..."
2,wavs/yo_p_0003.wav,"Kò fẹ́ẹ̀ lè dá dúró, mo kàn ní kí n su..."
3,wavs/yo_p_0004.wav,"Mo ń rìnrìn àjò káàkiri ni, lọ síbikí..."
4,wavs/yo_p_0005.wav,"Ìgbà ẹ̀ẹ̀rùn ti fẹ́ parí, oòrùn ti wọ̀ t..."
...,...,...
4245,wavs/yo_p_4246.wav,Ọba ló ba lórí ohun gbogbo ní àwùjọ tí ó bá tí...
4246,wavs/yo_p_4247.wav,Ṣé ẹni tí ó bá láyà nìkan ni ó lè mọ òwe Yorùb...
4247,wavs/yo_p_4248.wav,Ó yẹ kí wọ́n ya ọjọ́ kan sọ́tọ̀ láti máa ṣe ìr...
4248,wavs/yo_p_4249.wav,Ọ̀rọ̀ àwọn ọmọ náà kò tiẹ̀ fẹ́ yé wa mọ́ rár...


In [15]:
path = "/home/mila/g/guzmand/scratch/datasets/IroyinSpeech/male/train_1_hours.tsv"
df = pd.read_csv(path, sep="\t", names=["audio_file", "text"])

samples_train = len(df)
batch_size = 16
steps_per_epoch = math.ceil(samples_train / batch_size)
total_steps = 100_000

total_epochs = total_steps // steps_per_epoch
print(f"Total epochs: {total_epochs}")

df["audio_file"] = "wavs/" + df["audio_file"]
df.to_csv("data/yoruba-1-hours/metadata.csv", index=False, sep="|")
df

Total epochs: 2083


,audio_file,text
0,wavs/yo_p_0001.wav,Mo pàdé alàgbà ìnàkí yìí nílé ìtur...
1,wavs/yo_p_0002.wav,"Ilé ìtura náà ti gbó, tàbí ká ní ó t..."
2,wavs/yo_p_0003.wav,"Kò fẹ́ẹ̀ lè dá dúró, mo kàn ní kí n su..."
3,wavs/yo_p_0004.wav,"Mo ń rìnrìn àjò káàkiri ni, lọ síbikí..."
4,wavs/yo_p_0005.wav,"Ìgbà ẹ̀ẹ̀rùn ti fẹ́ parí, oòrùn ti wọ̀ t..."
...,...,...
752,wavs/yo_p_0753.wav,Etí ì mi ò kọbiara púpọ̀ tó bẹ́ẹ̀ sí kí...
753,wavs/yo_p_0754.wav,Báyìí ni gbogbo ohun tí ó ń dún ṣe mú ...
754,wavs/yo_p_0755.wav,A jókòó lóríi àpótí tí ọmọ ọ̀dọ̀ mi t...
755,wavs/yo_p_0756.wav,Ọmọ yẹn ṣá ti kọ́ bí wọ́n ṣe ń gbálẹ́ báy...


In [16]:
path = "/home/mila/g/guzmand/scratch/datasets/IroyinSpeech/male/train_30_minutes.tsv"
df = pd.read_csv(path, sep="\t", names=["audio_file", "text"])

samples_train = len(df)
batch_size = 16
steps_per_epoch = math.ceil(samples_train / batch_size)
total_steps = 100_000

total_epochs = total_steps // steps_per_epoch
print(f"Total epochs: {total_epochs}")

df["audio_file"] = "wavs/" + df["audio_file"]
df.to_csv("data/yoruba-30-minutes/metadata.csv", index=False, sep="|")
df

Total epochs: 4347


,audio_file,text
0,wavs/yo_p_0001.wav,Mo pàdé alàgbà ìnàkí yìí nílé ìtur...
1,wavs/yo_p_0002.wav,"Ilé ìtura náà ti gbó, tàbí ká ní ó t..."
2,wavs/yo_p_0003.wav,"Kò fẹ́ẹ̀ lè dá dúró, mo kàn ní kí n su..."
3,wavs/yo_p_0004.wav,"Mo ń rìnrìn àjò káàkiri ni, lọ síbikí..."
4,wavs/yo_p_0005.wav,"Ìgbà ẹ̀ẹ̀rùn ti fẹ́ parí, oòrùn ti wọ̀ t..."
...,...,...
356,wavs/yo_p_0357.wav,Ẹ ti ń fojú sọ́nà láti bẹ̀ẹ̀rẹ̀ sí ní kópa.
357,wavs/yo_p_0358.wav,Eléyìí fara hàn pé ẹ ti gbáradì lọ́pọ̀lọpọ̀ fú...
358,wavs/yo_p_0359.wav,Ìrètí ààrẹ Abíọ́lá ni láti ríi pé orúkọ Nàì...
359,wavs/yo_p_0360.wav,Ó yẹ kí orílẹ̀-èdè wa sì fakọyọ nínú ìdíje ná...


In [4]:
from datasets import Dataset
import pandas as pd

# Path to your file
path = "/home/mila/g/guzmand/scratch/Repositories/F5-TTS/data/yoruba-30-minutes_pinyin/raw.arrow"

# Load the dataset from Hugging Face Arrow file
dataset = Dataset.from_file(path)

# Extract columns from the single row (each is a list of entries)
audio_paths = dataset["audio_path"]
texts = dataset["text"]
durations = dataset["duration"]

# Convert to a pandas DataFrame
df = pd.DataFrame({
    "audio_path": audio_paths,
    "text": ["".join(t) for t in texts],
    "duration": durations
})

df


,audio_path,text,duration
0,data/yoruba-30-minutes/wavs/yo_p_0001.wav,Mo pà dé alà gbà ì nà kí yìí ní lé ...,5.086667
1,data/yoruba-30-minutes/wavs/yo_p_0002.wav,"Ilé ì tura náà ti gbó, tà bí ká ní ó...",3.813333
2,data/yoruba-30-minutes/wavs/yo_p_0003.wav,"Kò fẹ́ẹ̀ lè dá dú ró, mo kàn ní kí n s...",4.660000
3,data/yoruba-30-minutes/wavs/yo_p_0004.wav,"Mo ń rì nrìn à jò káà kiri ni, lọ sí b...",4.753333
4,data/yoruba-30-minutes/wavs/yo_p_0005.wav,"Ì gbà ẹ̀ẹ̀ rùn ti fẹ́ parí, oò rùn ti wọ...",4.233333
...,...,...,...
356,data/yoruba-30-minutes/wavs/yo_p_0357.wav,Ẹ ti ń fojú sọ́nà lá ti bẹ̀ẹ̀rẹ̀ sí ní kó pa.,3.413333
357,data/yoruba-30-minutes/wavs/yo_p_0358.wav,Eléyìí fara hàn pé ẹ ti gbá radì lọ́pọ̀lọpọ̀ f...,5.770159
358,data/yoruba-30-minutes/wavs/yo_p_0359.wav,Ìrètí ààrẹ Abíọ́ lá ni lá ti ríi pé orúkọ N...,5.096780
359,data/yoruba-30-minutes/wavs/yo_p_0360.wav,Ó yẹ kí orílẹ̀-èdè wa sì fakọyọ nínú ìdí je n...,4.585941


In [1]:
from f5_tts.model.dataset import load_dataset

mel_spec_kwargs = dict(
    target_sample_rate=24000,
    n_mel_channels=100,
    hop_length=256,
    win_length=1024,
    n_fft=1024,
    mel_spec_type="vocos"
)

train_dataset = load_dataset("yoruba-30-minutes", "pinyin", mel_spec_kwargs=mel_spec_kwargs)

/home/mila/g/guzmand/scratch/.conda/envs/F5-TTS/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/mila/g/guzmand/scratch/.conda/envs/F5-TTS/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset ...


In [2]:
len(train_dataset)

361

In [ ]:
python src/f5_tts/train/datasets/prepare_csv_wavs.py data/yoruba-5-hours data/yoruba-5-hours_pinyin --workers 4
python src/f5_tts/train/datasets/prepare_csv_wavs.py data/yoruba-1-hours data/yoruba-1-hours_pinyin --workers 4
python src/f5_tts/train/datasets/prepare_csv_wavs.py data/yoruba-30-minutes data/yoruba-30-minutes_pinyin --workers 4

In [ ]:
accelerate launch src/f5_tts/train/train.py --config-name F5TTS_v1_Base_5_hours.yaml
accelerate launch src/f5_tts/train/train.py --config-name F5TTS_v1_Base_1_hours.yaml
accelerate launch src/f5_tts/train/train.py --config-name F5TTS_v1_Base_30_minutes.yaml
